# Candidate Recommendation — Baseline C1 (TF-IDF + Cosine)
Query: **Job description → Top-K candidate CVs**

Datasets expected (relative to this notebook):
- `../datasets/it_jobs.xlsx`
- `../datasets/UpdatedResumeDataSet.csv`


In [1]:
import os, re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [2]:
# Paths (adjust if your folder layout differs)
JOBS_PATH = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/it_jobs.xlsx"
RESUMES_PATH = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv"

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower().replace("\n", " ").replace("\r", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

jobs = pd.read_excel(JOBS_PATH)
resumes = pd.read_csv(RESUMES_PATH)

print("Jobs:", jobs.shape, "Resumes:", resumes.shape)
print("Job columns:", list(jobs.columns)[:20])
print("Resume columns:", list(resumes.columns))


Jobs: (21961, 35) Resumes: (962, 2)
Job columns: ['#', 'Unnamed: 0', 'id', 'site', 'job_url', 'job_url_direct', 'title', 'company', 'location', 'job_type', 'date_posted', 'salary_source', 'interval', 'min_amount', 'max_amount', 'currency', 'is_remote', 'job_level', 'job_function', 'company_industry']
Resume columns: ['Category', 'Resume']


## Fit TF-IDF on resume corpus (run once)
This builds the TF-IDF matrix for all resumes so you can query multiple jobs quickly.

In [3]:
resume_texts = resumes["Resume"].fillna("").apply(clean_text)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=120_000
)

X_resumes = vectorizer.fit_transform(resume_texts)
print("TF-IDF resumes matrix:", X_resumes.shape)


TF-IDF resumes matrix: (962, 44314)


## Recommend candidates for one job index

In [5]:
def recommend_candidates_tfidf(job_index: int, top_k: int = 10):
    title = str(jobs.loc[job_index, "title"] if "title" in jobs.columns else "")
    desc = str(jobs.loc[job_index, "cleaned_description"] if "cleaned_description" in jobs.columns else "")
    job_text = clean_text(title + " " + desc)

    x_job = vectorizer.transform([job_text])
    scores = cosine_similarity(x_job, X_resumes).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]

    out = pd.DataFrame({
        "cv_index": top_idx,
        "score": scores[top_idx],
        "Category": resumes.loc[top_idx, "Category"].values if "Category" in resumes.columns else None,
        "resume_preview": resumes.loc[top_idx, "Resume"].apply(lambda s: str(s)[:140] + "...")
    })

    out.insert(0, "job_index", job_index)
    out.insert(1, "job_title", jobs.loc[job_index, "title"] if "title" in jobs.columns else "")
    return out

# Try one job
recommend_candidates_tfidf(job_index=0, top_k=10)


,job_index,job_title,cv_index,score,Category,resume_preview
547,0,Support Technologist II 2024-02216,547,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
539,0,Support Technologist II 2024-02216,539,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
515,0,Support Technologist II 2024-02216,515,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
519,0,Support Technologist II 2024-02216,519,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
511,0,Support Technologist II 2024-02216,511,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
535,0,Support Technologist II 2024-02216,535,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
523,0,Support Technologist II 2024-02216,523,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
531,0,Support Technologist II 2024-02216,531,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
527,0,Support Technologist II 2024-02216,527,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
543,0,Support Technologist II 2024-02216,543,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...


## Run a small experiment (multiple jobs) and export CSV

In [6]:
os.makedirs("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/employer/candrec_tfidf", exist_ok=True)

job_indices = [0, 10, 50]  # change as you like
top_k = 10

all_rows = []
for j in job_indices:
    all_rows.append(recommend_candidates_tfidf(j, top_k=top_k))

results = pd.concat(all_rows, ignore_index=True)
out_path = f"/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/employer/candrec_tfidf/candrec_tfidf_jobs{len(job_indices)}_top{top_k}.csv"
results.to_csv(out_path, index=False)
print("Saved:", out_path)

results.head(15)


Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/employer/candrec_tfidf/candrec_tfidf_jobs3_top10.csv


,job_index,job_title,cv_index,score,Category,resume_preview
0,0,Support Technologist II 2024-02216,547,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
1,0,Support Technologist II 2024-02216,539,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
2,0,Support Technologist II 2024-02216,515,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
3,0,Support Technologist II 2024-02216,519,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
4,0,Support Technologist II 2024-02216,511,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
5,0,Support Technologist II 2024-02216,535,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
6,0,Support Technologist II 2024-02216,523,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
7,0,Support Technologist II 2024-02216,531,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
8,0,Support Technologist II 2024-02216,527,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
9,0,Support Technologist II 2024-02216,543,0.423833,Operations Manager,Education Details \r\n BCA Vinayaka Missions...
